In [6]:
import re
import gurobipy as gp
from gurobipy import GRB

In [7]:
#NETWORK
network_file_path = "network.txt"

distance = {} #distance definition (from i to j)
nodes = set() #node set (all nodes)

#read network file
with open(network_file_path, "r") as network_file:
    #reading line by line
    for line in network_file:
        #removing extra spaces/line breaks
        line = line.strip()
        #skipping empty lines (safety check)
        if line == "":
            continue
        
        #splitting the line into three parts: i, j, d
        origin, destination, dist = line.split()
        
        #converting txt to numbres
        origin = int(origin)
        destination = int(destination)
        dist = float(dist)
        
        #noting distance between each OD pair
        distance[(origin, destination)] = dist
        
        #keep track of all nodes
        nodes.add(origin)
        nodes.add(destination)

print("Number of nodes found:", len(nodes))
print("Example distance entry:", list(distance.items())[:3])

Number of nodes found: 40
Example distance entry: [((0, 4), 291.0), ((4, 0), 291.0), ((0, 23), 99.0)]


In [8]:
#ROUTES
routes_file_path = "routes.txt"

routes_customers = []   # list of customers visited (WITHOUT depot)
route_distance  = []    # list of distance (cost) of each route

with open(routes_file_path, "r") as routes_file:
    for line in routes_file:
        line = line.strip()
        if line == "":
            continue
        
        numbers = [int(x) for x in re.findall(r"-?\d+", line)]
        
        #safety check: must have at least: distance, load, 0, 0
        if len(numbers) < 4:
            continue
        
        dist = numbers[0]          # first column: distance
        load = numbers[1]          # second column: load (not needed for 1a)
        path = numbers[2:]         # the route itself: starts/ends with depot 0
        
        #remove depot 0 if present at start/end
        if path[0] == 0:
            path = path[1:]
        if path and path[-1] == 0:
            path = path[:-1]
        
        routes_customers.append(tuple(path))
        route_distance.append(float(dist))

print("Number of routes read:", len(routes_customers))
print("First 3 routes (customers only):", routes_customers[:3])
print("First 3 route distances:", route_distance[:3])

Number of routes read: 14349
First 3 routes (customers only): [(14,), (1,), (6,)]
First 3 route distances: [932.0, 790.0, 1282.0]


In [9]:
#defining SETS
K = range(5) #number of vehicles given in assignment
R = range(len(routes_customers)) #number of feasible routes
customers = set()
for route in routes_customers:
    for i in route:
        customers.add(i)
customers = sorted(customers)

print("Nr of customers:", len(customers))
print("Customers:", customers)
print(range(len(routes_customers)))

Nr of customers: 25
Customers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
range(0, 14349)


In [10]:
#building the actual model
model = gp.Model("CVRP based cost minimization (1a)")

#decision bariable (4f) if vehicle k chooses route r {0,1} 
x = model.addVars(K, R, vtype=GRB.BINARY, name="x")

In [11]:
#objective cost minimzation
    #with the assumption that route distance equals the cost!!!!!!!
model.setObjective(
    gp.quicksum(route_distance[r] *x[k,r] for k in K for r in R),
    GRB.MINIMIZE
)

In [12]:
#customer coverage constraint (4b) - each customer served once
for i in customers:
    model.addConstr(
        gp.quicksum(
            x[k,r] for k in K for r in R if i in routes_customers[r] ) ==1,
        name=f"cover_customer_{i}"
    )

In [13]:
#vehicle constraint (4d) - one route per vehicle
for k in K:
    model.addConstr(
        gp.quicksum(x[k,r] for r in R) == 1,
        name=f"one_route_per_verhicle_{k}"
    )

In [14]:
#solving the model
model.optimize()

#RESULTS
if model.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    chosen_routes = []

    for k in K:
        for r in R:
            if x[k, r].X > 0.5:
                chosen_routes.append(r)
                print(
                    f"Vehicle {k} → route {r}, "
                    f"customers {routes_customers[r]}, "
                    f"distance {route_distance[r]}"
                )

    costs = [route_distance[r] for r in chosen_routes]

    print("\nSummary:")
    print("Total cost:", sum(costs))
    print("Max route distance:", max(costs))
    print("Min route distance:", min(costs))
    print("Range (max − min):", max(costs) - min(costs))
    print("Runtime (s):", model.Runtime)
else:
    print("No fesible solution found.")


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 30 rows, 71745 columns and 493020 nonzeros
Model fingerprint: 0x3b89ed47
Variable types: 0 continuous, 71745 integer (71745 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+02, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 0 rows and 42815 columns
Presolve time: 0.33s
Presolved: 30 rows, 28930 columns, 197780 nonzeros
Variable types: 0 continuous, 28930 integer (28930 binary)

Root relaxation: objective 8.703200e+03, 372 iterations, 0.20 seconds (0.37 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 8703.20000  

In [15]:
#PART 1B -------------------------------------------------------------
epsilon = 0.05
z_star = 8831.0

K = range(5) #(repitiion)
R = range(len(routes_customers)) #feasible routes


In [16]:
#MAKING MODEL 1B
model_1b_LP = gp.Model("Q1b_LP_relaxation")

#using y instead of x so that the jupyter doesn't accidentally take the wrong one
y = model_1b_LP.addVars(K, R, vtype=GRB.CONTINUOUS, name="y")

#add fairness variables
gamma = model_1b_LP.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="gamma")
eta = model_1b_LP.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="eta")

model_1b_LP.setObjective(eta - gamma, GRB.MINIMIZE) #minimize range eta - gamma

In [17]:
#customer coverage constraint (4b)
for i in customers:
    model_1b_LP.addConstr(
        gp.quicksum(y[k,r] for k in K for r in R if i in routes_customers[r]) ==1,
        name=f"cover_{i}"
    )

#vehicle route constraint (4d)
for k in K:
    model_1b_LP.addConstr(
        gp.quicksum(y[k,r] for r in R) ==1,
        name=f"one_route_{k}"
        )

#efficiency constraint (cost should be lower than (1+e)z*) aka (105% requirement)
model_1b_LP.addConstr(
    gp.quicksum(route_distance[r] * y[k,r] for k in K for r in R) <= ((1+epsilon)*z_star),
    name=f"cost_cap"
)

<gurobi.Constr *Awaiting Model Update*>

In [18]:
#DEFINING PAYOFFS
#payoff vehicle k is distance of its chosen route
for k in K:
    payoff_k =  gp.quicksum(route_distance[r] * y[k,r] for r in R)
    model_1b_LP.addConstr(gamma <= payoff_k, name=f"gamma_def_{k}")
    model_1b_LP.addConstr(payoff_k <= eta, name=f"eta_def_{k}")

In [19]:
#SOLVING THE MODEL
model_1b_LP.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 41 rows, 71747 columns and 708265 nonzeros
Model fingerprint: 0x65fad0f3
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 9e+03]
Presolve removed 0 rows and 5015 columns
Presolve time: 0.19s
Presolved: 41 rows, 66732 columns, 661215 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 7.400e+02
 Factor NZ  : 8.610e+02 (roughly 27 MB of memory)
 Factor Ops : 2.382e+04 (less than 1 second per iteration)
 Threads    : 4

                  Objective                Residual
Iter       Primal          Dual         Prim

In [21]:
if model_1b_LP.SolCount > 0:
    chosen = []
    costs = []
    
    print("\nChosen routes (vehicle-index):")
    for k in K:
        for r in R:
            if y[k, r].X > 0.5:
                chosen.append((k, r))
                costs.append(route_distance[r])
                print(f"Vehicle {k}: route {r} | customers {routes_customers[r]} | dist {route_distance[r]}")

    print("\nSummary (1b):")
    print("Total cost:", sum(costs))
    print("Cost cap:", (1.0 + epsilon) * z_star)
    print("Max route distance:", max(costs))
    print("Min route distance:", min(costs))
    print("Range (max-min):", max(costs) - min(costs))
    print("gamma:", gamma.X)
    print("eta:", eta.X)
    print("eta-gamma:", eta.X - gamma.X)
    print("Runtime (s):", model_1b_LP.Runtime)
else:
    print("No feasible solution found within the time limit (this does NOT necessarily mean infeasible).")


Chosen routes (vehicle-index):
Vehicle 4: route 6501 | customers (13, 20, 16, 12, 17, 4) | dist 2584.0

Summary (1b):
Total cost: 2584.0
Cost cap: 9272.550000000001
Max route distance: 2584.0
Min route distance: 2584.0
Range (max-min): 0.0
gamma: 1833.8064422216066
eta: 1833.8064422216066
eta-gamma: 0.0
Runtime (s): 0.690000057220459


In [23]:
#CORRECTNESS CHECK!!!!!!!!
#checking whether each vehicle picks one route 
for k in K:
    print("vehicle picks route", k, sum(y[k,r].X for r in R))

print("-----")
#is customer covered once?
for i in customers:
    cover = sum(y[k,r].X for k in K for r in R if i in routes_customers[r])
    if abs(cover - 1) > 1e-6:
        print("Customer coverage issue:", i, cover)
        break
else:
    print("All customers covered exactly once.")

print("-----")
#is cost cap respected?
total_cost = sum(route_distance[r] * y[k,r].X for k in K for r in R)
print("Total cost:", total_cost)
print("Cap:", 1.05 * z_star)

print("-----")
#does range match eta-gamma?
payoffs = []
for k in K:
    payoffs.append(sum(route_distance[r] * y[k,r].X for r in R))
print("Payoffs:", payoffs)
print("Range (max-min):", max(payoffs) - min(payoffs))
print("eta-gamma:", eta.X - gamma.X)


vehicle picks route 0 1.0
vehicle picks route 1 1.0
vehicle picks route 2 1.0
vehicle picks route 3 1.0
vehicle picks route 4 1.0
-----
All customers covered exactly once.
-----
Total cost: 9169.032211108033
Cap: 9272.550000000001
-----
Payoffs: [1833.806442221607, 1833.8064422216064, 1833.8064422216066, 1833.8064422216073, 1833.8064422216069]
Range (max-min): 9.094947017729282e-13
eta-gamma: 0.0
